In [1]:
import torch

In [58]:
# pt2_path = "../model/transformer_exported.pt2"
pt2_path = "../model/transformer_tp_rank1.pt2"

exported_program = torch.export.load(pt2_path)

help(exported_program)

Help on ExportedProgram in module torch.export.exported_program object:

class ExportedProgram(builtins.object)
 |  ExportedProgram(root: torch.nn.modules.module.Module | dict[str, typing.Any], graph: torch.fx.graph.Graph, graph_signature: torch.export.graph_signature.ExportGraphSignature, state_dict: dict[str, torch.Tensor | torch.nn.parameter.Parameter], range_constraints: 'dict[sympy.Symbol, Any]', module_call_graph: list[torch.export.exported_program.ModuleCallEntry], example_inputs: tuple[tuple[typing.Any, ...], dict[str, typing.Any]] | None = None, constants: dict[str, typing.Union[torch.Tensor, torch.ScriptObject, torch._library.fake_class_registry.FakeScriptObject, torch.utils._pytree.TreeSpec]] | None = None, *, verifiers: list[type[torch._export.verifier.Verifier]] | None = None)
 |  
 |  Package of a program from :func:`export`. It contains
 |  an :class:`torch.fx.Graph` that represents Tensor computation, a state_dict containing
 |  tensor values of all lifted parameters an

In [59]:
# signature
signature = exported_program.graph_signature

help(signature)

print(f"signature.user_inputs: {signature.user_inputs}")
print(f"signature.user_outputs: {signature.user_outputs}")
print(f"signature.parameters: {signature.parameters}")

Help on ExportGraphSignature in module torch.export.graph_signature object:

class ExportGraphSignature(builtins.object)
 |  ExportGraphSignature(input_specs: list[torch.export.graph_signature.InputSpec], output_specs: list[torch.export.graph_signature.OutputSpec]) -> None
 |  
 |  :class:`ExportGraphSignature` models the input/output signature of Export Graph,
 |  which is a fx.Graph with stronger invariants guarantees.
 |  
 |  Export Graph is functional and does not access "states" like parameters
 |  or buffers within the graph via ``getattr`` nodes. Instead, :func:`export`
 |  guarantees that parameters, buffers, and constant tensors are lifted out of
 |  the graph as inputs.  Similarly, any mutations to buffers are not included
 |  in the graph either, instead the updated values of mutated buffers are
 |  modeled as additional outputs of Export Graph.
 |  
 |  The ordering of all inputs and outputs are::
 |  
 |      Inputs = [*parameters_buffers_constant_tensors, *flattened_user

In [60]:
# graph
graph_module = exported_program.graph_module
graph = graph_module.graph

print(f"total nodes: {len(graph.nodes)}")

for node in graph.nodes:
    help(node)
    break

for node in graph.nodes:
    print(f"node.name: {node.name:<20} node.op: {node.op:<20} node.target: {node.target}")
    args = node.args
    print(f"    number of args: {len(args)}")
    for i,arg in enumerate(args):
        print(f"        index:{i} arg: {arg} type of arg: {type(arg)}")
    print("-"*80)

total nodes: 54
Help on Node in module torch.fx.node object:

class Node(torch._C._NodeBase)
 |  Node(graph: 'Graph', name: str, op: str, target: 'Target', args: tuple['Argument', ...], kwargs: dict[str, 'Argument'], return_type: Optional[Any] = None) -> None
 |  
 |  ``Node`` is the data structure that represents individual operations within
 |  a ``Graph``. For the most part, Nodes represent callsites to various entities,
 |  such as operators, methods, and Modules (some exceptions include nodes that
 |  specify function inputs and outputs). Each ``Node`` has a function specified
 |  by its ``op`` property. The ``Node`` semantics for each value of ``op`` are as follows:
 |  
 |  - ``placeholder`` represents a function input. The ``name`` attribute specifies the name this value will take on.
 |    ``target`` is similarly the name of the argument. ``args`` holds either: 1) nothing, or 2) a single argument
 |    denoting the default parameter of the function input. ``kwargs`` is don't-c

In [61]:
# state_dict
state_dict = exported_program.state_dict

for name,param in state_dict.items():
    print(f"name: {name:<20} param:{type(param)}   param.shape:{param.shape}")

name: embedding.weight     param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([1000, 256])
name: q_proj.weight        param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([128, 256])
name: k_proj.weight        param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([128, 256])
name: v_proj.weight        param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([128, 256])
name: o_proj.weight        param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([256, 128])
name: fc1.weight           param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([512, 256])
name: fc1.bias             param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([512])
name: fc2.weight           param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([256, 512])
name: fc2.bias             param:<class 'torch.nn.parameter.Parameter'>   param.shape:torch.Size([256])
name: ln1.weight           p